# Week 02: Electric Potential and Energy -- Scalar Fields and Energy Conservation
## PHASE 1: Electric Fields & Energy

*Physics II (PHY102) . 3 Hours . Dr. Arif Solmaz*

## Learning Objectives

By the end of this week, you will be able to:

- Define electric potential and explain why it is a scalar quantity (simpler than the vector field)
- Calculate the electric potential due to one or more point charges at any location in space
- Explain the relationship between the electric field and the electric potential ($\vec{E} = -\nabla V$)
- Draw and interpret equipotential surfaces and their relationship to electric field lines
- Calculate the work done by the electric force when moving a charge between two points
- Apply energy conservation (kinetic + potential energy) to solve problems with moving charges
- Visualize the potential landscape as a 3D surface and connect it to the physics of charges

## 🎯 Core Mastery Connection

Potential is the energy map of the electric field. This week you learn to predict how much energy a charge gains or loses as it moves through a field. The scalar potential $V$ is often easier to work with than the vector field $\vec{E}$, and the relationship $\vec{E} = -\nabla V$ connects the two. Mastering potential means you can predict particle speeds, energy transfers, and voltage differences in any configuration.

> **Framework for every problem:** Configuration → Law → Equation → Prediction → Verify.

> **From Physics I:** Before starting this week, make sure you are comfortable with:
> - The work-energy theorem ($W = \vec{F} \cdot \vec{d}$) and the concept of work done by a force
> - Conservative forces and potential energy (gravitational PE as the model case)
> - Line integrals at a conceptual level — work as the integral of force along a path

In [ ]:
# ============================================================
# Setup: Import all required libraries
# ============================================================
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import FloatSlider, IntSlider, interact, interactive, HBox, VBox, Layout

# Set default plot style
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

# Constants
k_e = 8.9875e9  # Coulomb constant, N m^2 / C^2
e_charge = 1.602e-19  # elementary charge, C

print("All libraries loaded successfully!")
print(f"Coulomb constant k = {k_e:.4e} N m^2/C^2")
print(f"Elementary charge e = {e_charge:.3e} C")

---
## 1. Electric Potential: From Vectors to Scalars

Last week we studied the electric field $\vec{E}$, which is a **vector** at every point in space. This week we introduce a simpler quantity: the **electric potential** $V$, which is a **scalar** at every point.

### Definition

The electric potential at a point is the **potential energy per unit charge**:

$$V = \frac{U}{q_0}$$

For a **point charge** $Q$, the potential at a distance $r$ is:

$$V = k_e \frac{Q}{r}$$

**Key differences from the electric field:**
- $V$ is a **scalar** (just a number, no direction)
- $V$ can be **positive or negative** (keeps the sign of $Q$)
- $V$ drops off as $1/r$ (not $1/r^2$ like $E$)
- **Units:** Volts (V) = Joules/Coulomb (J/C)

### Superposition

Because $V$ is a scalar, superposition is much simpler -- just add numbers:

$$V_{\text{total}} = \sum_i V_i = \sum_i k_e \frac{q_i}{r_i}$$

No vector components needed!

In [ ]:
# ============================================================
# Helper functions for potential and field calculations
# ============================================================

def V_point_charge(q, xq, yq, x, y):
    """Electric potential at (x, y) from charge q at (xq, yq)."""
    r = np.sqrt((x - xq)**2 + (y - yq)**2)
    r = np.where(r < 0.01, 0.01, r)  # avoid singularity
    return k_e * q / r

def V_total(charges, x, y):
    """Total potential from a list of charges [(q, xq, yq), ...]."""
    V = np.zeros_like(x, dtype=float)
    for q, xq, yq in charges:
        V += V_point_charge(q, xq, yq, x, y)
    return V

def E_field_point_charge(q, xq, yq, x, y):
    """E-field components at (x, y) from charge q at (xq, yq)."""
    dx = x - xq
    dy = y - yq
    r = np.sqrt(dx**2 + dy**2)
    r = np.where(r < 0.01, 0.01, r)
    E_mag = k_e * q / r**2
    return E_mag * dx / r, E_mag * dy / r

def E_field_total(charges, x, y):
    """Total E-field components from a list of charges."""
    Ex = np.zeros_like(x, dtype=float)
    Ey = np.zeros_like(y, dtype=float)
    for q, xq, yq in charges:
        ex, ey = E_field_point_charge(q, xq, yq, x, y)
        Ex += ex
        Ey += ey
    return Ex, Ey

print("Helper functions defined: V_point_charge, V_total, E_field_point_charge, E_field_total")

In [ ]:
# ============================================================
# Interactive: Compare V(r) vs E(r) for a point charge
# ============================================================

def compare_V_and_E(Q=3.0):
    """
    Compare how V and E fall off with distance for a point charge.
    Q in microcoulombs.
    """
    Q_C = Q * 1e-6
    r = np.linspace(0.1, 3.0, 300)
    
    V_vals = k_e * Q_C / r           # V = kQ/r
    E_vals = k_e * abs(Q_C) / r**2   # E = k|Q|/r^2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: V(r)
    ax1.plot(r, V_vals / 1e3, 'b-', linewidth=2.5)
    ax1.axhline(y=0, color='gray', linewidth=0.8)
    ax1.set_xlabel('Distance r (m)', fontsize=12)
    ax1.set_ylabel('Potential V (kV)', fontsize=12)
    ax1.set_title(f'Electric Potential: V = kQ/r  (Q = {Q:.1f} uC)', fontsize=13)
    if Q == 0:
        ax1.set_ylim(-1, 1)
    else:
        V_min = V_vals.min() / 1e3
        V_max = V_vals.max() / 1e3
        margin = max(abs(V_min), abs(V_max)) * 0.1 + 1
        ax1.set_ylim(V_min - margin, V_max + margin)
    ax1.text(1.5, V_vals[150]/1e3 * 0.7 if Q != 0 else 0, r'$V \propto 1/r$', fontsize=14, color='blue')
    
    # Right: E(r)
    ax2.plot(r, E_vals / 1e3, 'r-', linewidth=2.5)
    ax2.set_xlabel('Distance r (m)', fontsize=12)
    ax2.set_ylabel('|E| (kN/C)', fontsize=12)
    ax2.set_title(f'Electric Field: E = k|Q|/r^2  (|Q| = {abs(Q):.1f} uC)', fontsize=13)
    if Q == 0:
        ax2.set_ylim(-0.1, 1)
    else:
        E_data_max = E_vals.max() / 1e3
        ax2.set_ylim(bottom=0, top=min(E_data_max * 0.5, 500))
    ax2.text(1.5, E_vals[150]/1e3 * 0.7 if Q != 0 else 0, r'$E \propto 1/r^2$', fontsize=14, color='red')
    
    plt.tight_layout()
    plt.show()
    
    if Q != 0:
        print(f"At r = 1.0 m:  V = {k_e * Q_C / 1.0:.2f} V,  E = {k_e * abs(Q_C) / 1.0**2:.2f} N/C")
        print(f"At r = 2.0 m:  V = {k_e * Q_C / 2.0:.2f} V,  E = {k_e * abs(Q_C) / 2.0**2:.2f} N/C")
        print(f"Notice: V drops by 1/2 when r doubles, but E drops by 1/4.")
    else:
        print("Q = 0: Both V and E are zero everywhere.")

interact(compare_V_and_E,
         Q=FloatSlider(min=-5, max=5, step=0.5, value=3.0,
                       description='Q (uC):', style={'description_width': 'initial'},
                       layout=Layout(width='500px')));

---
## 2. The Potential Landscape: 3D Surface Plot

One of the most intuitive ways to visualize electric potential is as a **3D surface**. Imagine the potential $V(x,y)$ as the "height" of a landscape:

- Positive charges create **peaks** (mountains)
- Negative charges create **valleys** (wells)
- A positive test charge "rolls downhill" on this landscape

This is similar to gravitational potential energy landscapes, where objects roll from high to low potential.

In [ ]:
# ============================================================
# Interactive Demo 1: 3D Potential Landscape
# ============================================================

def potential_landscape_3d(q1=3.0, x1=-1.0, y1=0.0,
                           q2=-2.0, x2=1.0, y2=0.0,
                           elevation=30, azimuth=45):
    """
    3D surface plot of V(x,y) from point charges.
    Charges in microcoulombs.
    """
    charges = []
    if q1 != 0:
        charges.append((q1 * 1e-6, x1, y1))
    if q2 != 0:
        charges.append((q2 * 1e-6, x2, y2))
    
    if len(charges) == 0:
        print("Set at least one charge to a nonzero value.")
        return
    
    # Create grid
    gx = np.linspace(-4, 4, 200)
    gy = np.linspace(-4, 4, 200)
    X, Y = np.meshgrid(gx, gy)
    
    # Calculate potential
    V = V_total(charges, X, Y)
    
    # Clip for better visualization (avoid singularity spikes)
    V_max = 5e4
    V_clip = np.clip(V, -V_max, V_max)
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Surface plot
    surf = ax.plot_surface(X, Y, V_clip, cmap='coolwarm',
                           alpha=0.85, rstride=3, cstride=3,
                           linewidth=0.1, edgecolor='gray')
    
    # Mark charge positions at the clipped potential value (on the surface)
    for q, xq, yq in charges:
        color = 'red' if q > 0 else 'blue'
        actual_V = k_e * q / 0.01
        z_val = np.clip(actual_V, -V_max, V_max)
        ax.scatter([xq], [yq], [z_val], color=color, s=100, zorder=10,
                   edgecolors='black', linewidth=2)
    
    ax.view_init(elev=elevation, azim=azimuth)
    ax.set_xlabel('x (m)', fontsize=11)
    ax.set_ylabel('y (m)', fontsize=11)
    ax.set_zlabel('V (V)', fontsize=11)
    ax.set_title('Electric Potential Landscape V(x, y)', fontsize=14)
    
    fig.colorbar(surf, ax=ax, shrink=0.5, label='V (Volts)')
    plt.tight_layout()
    plt.show()
    
    print("Positive charges create peaks (mountains). Negative charges create valleys (wells).")
    print("A positive test charge would roll downhill on this landscape.")

style = {'description_width': 'initial'}
layout = Layout(width='450px')

interact(potential_landscape_3d,
         q1=FloatSlider(min=-5, max=5, step=0.5, value=3.0,
                        description='q1 (uC):', style=style, layout=layout),
         x1=FloatSlider(min=-3, max=3, step=0.25, value=-1.0,
                        description='x1 (m):', style=style, layout=layout),
         y1=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y1 (m):', style=style, layout=layout),
         q2=FloatSlider(min=-5, max=5, step=0.5, value=-2.0,
                        description='q2 (uC):', style=style, layout=layout),
         x2=FloatSlider(min=-3, max=3, step=0.25, value=1.0,
                        description='x2 (m):', style=style, layout=layout),
         y2=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y2 (m):', style=style, layout=layout),
         elevation=IntSlider(min=0, max=90, step=5, value=30,
                             description='View elevation:', style=style, layout=layout),
         azimuth=IntSlider(min=0, max=360, step=10, value=45,
                           description='View azimuth:', style=style, layout=layout));

---
## 3. Equipotential Surfaces and Field Lines

**Equipotential surfaces** (or lines in 2D) are surfaces where $V$ is constant. Key properties:

1. Equipotential lines are always **perpendicular** to electric field lines
2. No work is done when moving a charge along an equipotential surface
3. The spacing between equipotentials indicates field strength (closer = stronger field)
4. Conductors in equilibrium are equipotential surfaces

### The E-V Relationship

The electric field is the **negative gradient** of the potential:

$$\vec{E} = -\nabla V = -\left(\frac{\partial V}{\partial x}\hat{x} + \frac{\partial V}{\partial y}\hat{y} + \frac{\partial V}{\partial z}\hat{z}\right)$$

This means:
- $\vec{E}$ points in the direction of **decreasing** potential
- The field is **strongest** where the potential changes most rapidly

In [ ]:
# ============================================================
# Interactive Demo 2: Equipotential Contour Map + Field Vectors
# ============================================================

def equipotential_map(q1=3.0, x1=-1.0, y1=0.0,
                      q2=-3.0, x2=1.0, y2=0.0,
                      n_contours=20, show_field_lines=True):
    """
    Show equipotential contours with electric field vectors overlaid.
    Charges in microcoulombs.
    """
    charges = []
    if q1 != 0:
        charges.append((q1 * 1e-6, x1, y1))
    if q2 != 0:
        charges.append((q2 * 1e-6, x2, y2))
    
    if len(charges) == 0:
        print("Set at least one charge to a nonzero value.")
        return
    
    # Fine grid
    gx = np.linspace(-4, 4, 200)
    gy = np.linspace(-4, 4, 200)
    X, Y = np.meshgrid(gx, gy)
    
    V = V_total(charges, X, Y)
    V_clip = np.clip(V, -5e4, 5e4)
    
    fig, ax = plt.subplots(figsize=(10, 9))
    
    # Equipotential contours
    # Use symmetric levels around 0
    V_max = min(np.percentile(np.abs(V_clip), 95), 5e4)
    levels = np.linspace(-V_max, V_max, n_contours)
    
    cs = ax.contour(X, Y, V_clip, levels=levels, cmap='coolwarm',
                    linewidths=1.5, alpha=0.8)
    ax.clabel(cs, inline=True, fontsize=7, fmt='%.0f V')
    
    # Filled contours for background
    ax.contourf(X, Y, V_clip, levels=50, cmap='coolwarm', alpha=0.3)
    
    # Field lines (streamplot)
    if show_field_lines:
        Ex, Ey = E_field_total(charges, X, Y)
        E_mag = np.sqrt(Ex**2 + Ey**2)
        lw = 1.5 * np.log1p(E_mag) / np.log1p(E_mag).max()
        ax.streamplot(X, Y, Ex, Ey, color='gray', density=1.5,
                      linewidth=lw, arrowsize=1.2, arrowstyle='->')
    
    # Draw charges
    for q, xq, yq in charges:
        color = 'red' if q > 0 else 'blue'
        ax.plot(xq, yq, 'o', color=color, markersize=18,
                markeredgecolor='black', markeredgewidth=2, zorder=10)
        sign = '+' if q > 0 else '-'
        ax.text(xq, yq, sign, ha='center', va='center',
                fontsize=14, fontweight='bold', color='white', zorder=11)
    
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)', fontsize=12)
    ax.set_ylabel('y (m)', fontsize=12)
    title = 'Equipotential Lines (colored) + Field Lines (gray)' if show_field_lines else 'Equipotential Lines'
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()
    
    if show_field_lines:
        print("Notice: Field lines (gray) are always PERPENDICULAR to equipotential lines (colored).")

style = {'description_width': 'initial'}
layout = Layout(width='450px')

interact(equipotential_map,
         q1=FloatSlider(min=-5, max=5, step=0.5, value=3.0,
                        description='q1 (uC):', style=style, layout=layout),
         x1=FloatSlider(min=-3, max=3, step=0.25, value=-1.0,
                        description='x1 (m):', style=style, layout=layout),
         y1=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y1 (m):', style=style, layout=layout),
         q2=FloatSlider(min=-5, max=5, step=0.5, value=-3.0,
                        description='q2 (uC):', style=style, layout=layout),
         x2=FloatSlider(min=-3, max=3, step=0.25, value=1.0,
                        description='x2 (m):', style=style, layout=layout),
         y2=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y2 (m):', style=style, layout=layout),
         n_contours=IntSlider(min=5, max=40, step=5, value=20,
                              description='Contour lines:', style=style, layout=layout),
         show_field_lines=widgets.Checkbox(value=True, description='Show field lines',
                                           style=style));

---
## 4. Work and Energy in Electric Fields

### Work Done by the Electric Force

The work done by the electric force in moving a charge $q_0$ from point A to point B is:

$$W_{A \to B} = q_0 (V_A - V_B) = -q_0 \Delta V$$

**Important points:**
- Work depends only on the **potential difference** $\Delta V = V_B - V_A$, not the path taken
- The electric force is **conservative** (path-independent)
- Moving a positive charge from high to low potential: the field does **positive** work
- Moving a positive charge from low to high potential: the field does **negative** work (you must push it)

### Electric Potential Energy

The potential energy of a charge $q_0$ at a point where the potential is $V$:

$$U = q_0 V$$

For two point charges $q_1$ and $q_2$ separated by distance $r$:

$$U = k_e \frac{q_1 q_2}{r}$$

### Energy Conservation

$$K_A + U_A = K_B + U_B$$
$$\frac{1}{2}mv_A^2 + q_0 V_A = \frac{1}{2}mv_B^2 + q_0 V_B$$

In [ ]:
# ============================================================
# Interactive Demo 4: Potential Difference Calculator
# ============================================================

def potential_difference_calc(Q=5.0, xQ=0.0, yQ=0.0,
                               xA=-2.0, yA=0.0,
                               xB=1.0, yB=1.5):
    """
    Calculate potential and potential difference between two points
    near a point charge. Q in microcoulombs.
    """
    Q_C = Q * 1e-6
    charges = [(Q_C, xQ, yQ)]
    
    # Distances
    rA = np.sqrt((xA - xQ)**2 + (yA - yQ)**2)
    rB = np.sqrt((xB - xQ)**2 + (yB - yQ)**2)
    
    rA = max(rA, 0.01)
    rB = max(rB, 0.01)
    
    VA = k_e * Q_C / rA
    VB = k_e * Q_C / rB
    delta_V = VB - VA
    
    # Work to move +1 uC from A to B
    q_test = 1e-6  # 1 uC test charge
    W = q_test * (VA - VB)  # work done by the field
    
    # Plot
    gx = np.linspace(-4, 4, 150)
    gy = np.linspace(-4, 4, 150)
    X, Y = np.meshgrid(gx, gy)
    V = V_total(charges, X, Y)
    V_clip = np.clip(V, -5e4, 5e4)
    
    fig, ax = plt.subplots(figsize=(9, 8))
    
    V_max = min(np.percentile(np.abs(V_clip), 90), 5e4)
    levels = np.linspace(-V_max, V_max, 25)
    cs = ax.contourf(X, Y, V_clip, levels=50, cmap='coolwarm', alpha=0.4)
    ax.contour(X, Y, V_clip, levels=levels, cmap='coolwarm', linewidths=1, alpha=0.6)
    plt.colorbar(cs, ax=ax, label='V (Volts)', shrink=0.8)
    
    # Charge
    c_color = 'red' if Q > 0 else 'blue'
    ax.plot(xQ, yQ, 'o', color=c_color, markersize=20,
            markeredgecolor='black', markeredgewidth=2, zorder=10)
    sign = '+' if Q > 0 else '-'
    ax.text(xQ, yQ, sign, ha='center', va='center', fontsize=16,
            fontweight='bold', color='white', zorder=11)
    
    # Points A and B
    ax.plot(xA, yA, 's', color='green', markersize=14, markeredgecolor='black',
            markeredgewidth=2, zorder=10, label=f'A ({xA:.1f}, {yA:.1f})')
    ax.text(xA + 0.15, yA + 0.15, f'A\nV={VA:.0f} V', fontsize=10,
            fontweight='bold', color='green')
    
    ax.plot(xB, yB, 'D', color='purple', markersize=14, markeredgecolor='black',
            markeredgewidth=2, zorder=10, label=f'B ({xB:.1f}, {yB:.1f})')
    ax.text(xB + 0.15, yB + 0.15, f'B\nV={VB:.0f} V', fontsize=10,
            fontweight='bold', color='purple')
    
    # Arrow from A to B
    ax.annotate('', xy=(xB, yB), xytext=(xA, yA),
               arrowprops=dict(arrowstyle='->', color='black', lw=2,
                              connectionstyle='arc3,rad=0.2'))
    
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title('Potential Difference Between Two Points', fontsize=13)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()
    
    print(f"Point A: r_A = {rA:.3f} m,  V_A = {VA:.2f} V")
    print(f"Point B: r_B = {rB:.3f} m,  V_B = {VB:.2f} V")
    print(f"Potential difference: DeltaV = V_B - V_A = {delta_V:.2f} V")
    print(f"")
    print(f"Work done by field to move q0 = +1 uC from A to B:")
    print(f"  W = q0 (V_A - V_B) = {W:.6f} J = {W*1e6:.2f} uJ")

style = {'description_width': 'initial'}
layout = Layout(width='450px')

interact(potential_difference_calc,
         Q=FloatSlider(min=-5, max=5, step=0.5, value=5.0,
                       description='Q (uC):', style=style, layout=layout),
         xQ=FloatSlider(min=-2, max=2, step=0.25, value=0.0,
                        description='xQ (m):', style=style, layout=layout),
         yQ=FloatSlider(min=-2, max=2, step=0.25, value=0.0,
                        description='yQ (m):', style=style, layout=layout),
         xA=FloatSlider(min=-3.5, max=3.5, step=0.25, value=-2.0,
                        description='xA (m):', style=style, layout=layout),
         yA=FloatSlider(min=-3.5, max=3.5, step=0.25, value=0.0,
                        description='yA (m):', style=style, layout=layout),
         xB=FloatSlider(min=-3.5, max=3.5, step=0.25, value=1.0,
                        description='xB (m):', style=style, layout=layout),
         yB=FloatSlider(min=-3.5, max=3.5, step=0.25, value=1.5,
                        description='yB (m):', style=style, layout=layout));

---
## 5. Worked Examples

In [ ]:
# ============================================================
# Worked Example 1: Potential at a Point from Two Charges
# ============================================================

print("=" * 60)
print("WORKED EXAMPLE 1: Potential at a Point")
print("=" * 60)
print()
print("Problem: Two charges are placed on the x-axis:")
print("  q1 = +4 uC at x = -0.20 m")
print("  q2 = -6 uC at x = +0.20 m")
print("Find the electric potential at the origin (0, 0).")
print()

q1 = 4e-6
q2 = -6e-6
x1, y1 = -0.20, 0.0
x2, y2 = 0.20, 0.0
xP, yP = 0.0, 0.0

r1 = np.sqrt((xP - x1)**2 + (yP - y1)**2)
r2 = np.sqrt((xP - x2)**2 + (yP - y2)**2)

V1 = k_e * q1 / r1
V2 = k_e * q2 / r2
V_total_val = V1 + V2

print("Solution:")
print(f"  r1 = distance from q1 to origin = {r1:.2f} m")
print(f"  r2 = distance from q2 to origin = {r2:.2f} m")
print()
print(f"  V1 = k q1/r1 = ({k_e:.3e})({q1:.1e})/{r1} = {V1:.2f} V")
print(f"  V2 = k q2/r2 = ({k_e:.3e})({q2:.1e})/{r2} = {V2:.2f} V")
print()
print(f"  V_total = V1 + V2 = {V1:.2f} + ({V2:.2f}) = {V_total_val:.2f} V")
print()
print("  Note: Potential is a SCALAR -- we just add numbers, no vectors needed!")
print(f"  The negative potential means V < 0 at the origin (the negative charge dominates).")

In [ ]:
# ============================================================
# Worked Example 2: Work Done Moving a Charge
# ============================================================

print("=" * 60)
print("WORKED EXAMPLE 2: Work Done Moving a Charge")
print("=" * 60)
print()
print("Problem: A charge Q = +8 uC is fixed at the origin.")
print("How much work is done by the electric field when")
print("a test charge q0 = +2 uC moves from r_A = 0.50 m")
print("to r_B = 1.50 m?")
print()

Q_val = 8e-6
q0 = 2e-6
rA = 0.50
rB = 1.50

VA = k_e * Q_val / rA
VB = k_e * Q_val / rB
W = q0 * (VA - VB)

print("Solution:")
print(f"  V_A = kQ/r_A = ({k_e:.3e})({Q_val:.1e})/{rA} = {VA:.2f} V")
print(f"  V_B = kQ/r_B = ({k_e:.3e})({Q_val:.1e})/{rB} = {VB:.2f} V")
print()
print(f"  Work by field: W = q0 (V_A - V_B)")
print(f"    = ({q0:.1e})({VA:.2f} - {VB:.2f})")
print(f"    = ({q0:.1e})({VA - VB:.2f})")
print(f"    = {W:.6f} J")
print(f"    = {W*1e3:.3f} mJ")
print()
print(f"  Since W > 0, the field does positive work.")
print(f"  This makes sense: both charges are positive, so the field")
print(f"  pushes q0 away from Q (from smaller r to larger r).")
print(f"  The charge moves 'downhill' in potential.")

# Visualization
fig, ax = plt.subplots(figsize=(10, 4))
r_arr = np.linspace(0.2, 3.0, 300)
V_arr = k_e * Q_val / r_arr

ax.plot(r_arr, V_arr / 1e3, 'b-', linewidth=2.5, label='V(r) = kQ/r')
ax.plot(rA, VA / 1e3, 'go', markersize=12, zorder=5, label=f'A: V = {VA/1e3:.1f} kV')
ax.plot(rB, VB / 1e3, 'rs', markersize=12, zorder=5, label=f'B: V = {VB/1e3:.1f} kV')
ax.annotate('', xy=(rB, VB / 1e3), xytext=(rA, VA / 1e3),
           arrowprops=dict(arrowstyle='->', color='orange', lw=2.5,
                          connectionstyle='arc3,rad=-0.3'))
ax.text((rA + rB)/2, (VA + VB)/2/1e3 * 0.7,
        f'W = {W*1e3:.3f} mJ', fontsize=12, color='orange', fontweight='bold')

# Shade the area
mask = (r_arr >= rA) & (r_arr <= rB)
ax.fill_between(r_arr[mask], 0, V_arr[mask] / 1e3, alpha=0.15, color='green')

ax.set_xlabel('Distance r (m)', fontsize=12)
ax.set_ylabel('V (kV)', fontsize=12)
ax.set_title('Work = charge moves from high V to low V', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

---
## 6. Animation: Test Charge Moving in a Potential Field

Let us watch a positive test charge move under the influence of a fixed charge. As it moves, we track:
- **Kinetic energy** $K = \frac{1}{2}mv^2$
- **Potential energy** $U = q_0 V$
- **Total energy** $E_{\text{total}} = K + U$ (should be constant!)

This demonstrates **conservation of energy** in electric fields.

In [ ]:
# ============================================================
# Animation: Test Charge Moving in a Potential Field
# Shows energy conservation (K + U = const)
# ============================================================

# Setup: Fixed positive charge at origin, test charge starts at (3, 0) moving toward it
Q_fixed = 5e-6   # Fixed charge (C)
q_test = 1e-6    # Test charge (C)
m_test = 1e-6    # Mass of test charge (kg) -- artificial for visualization

# Initial conditions
x0 = 3.0       # starting x position (m)
v0 = -2.0      # initial velocity (m/s), moving toward origin

# Simulate the motion using Euler's method
dt = 0.005
n_steps = 600

x_traj = np.zeros(n_steps)
v_traj = np.zeros(n_steps)
K_traj = np.zeros(n_steps)
U_traj = np.zeros(n_steps)

x_traj[0] = x0
v_traj[0] = v0

for i in range(n_steps - 1):
    r = abs(x_traj[i])
    r = max(r, 0.2)  # prevent getting too close
    
    # Force: F = k q_test Q_fixed / r^2, direction: away from origin if same sign
    F = k_e * q_test * Q_fixed / r**2
    if x_traj[i] < 0:
        F = -F  # force direction
    
    a = F / m_test
    v_traj[i + 1] = v_traj[i] + a * dt
    x_traj[i + 1] = x_traj[i] + v_traj[i + 1] * dt
    
    # Bounce off if too close
    if abs(x_traj[i + 1]) < 0.2:
        v_traj[i + 1] = -v_traj[i + 1]
        x_traj[i + 1] = np.sign(x_traj[i + 1]) * 0.2

# Compute energies
for i in range(n_steps):
    r = max(abs(x_traj[i]), 0.2)
    K_traj[i] = 0.5 * m_test * v_traj[i]**2
    U_traj[i] = k_e * q_test * Q_fixed / r

E_total = K_traj + U_traj

# Create animation
fig_anim, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8),
                                     gridspec_kw={'height_ratios': [1.2, 1]})

# Top: position diagram
ax1.set_xlim(-1, 4)
ax1.set_ylim(-1, 1)
ax1.set_aspect('equal')
ax1.set_xlabel('x (m)')
ax1.set_title('Test Charge Moving Near a Fixed Charge')

# Fixed charge
ax1.plot(0, 0, 'ro', markersize=20, zorder=10)
ax1.text(0, 0, '+', ha='center', va='center', fontsize=14,
         fontweight='bold', color='white', zorder=11)
ax1.text(0, -0.5, 'Q (fixed)', ha='center', fontsize=10)

test_dot, = ax1.plot([], [], 'go', markersize=14, zorder=10)
test_label = ax1.text(0, 0.5, '', fontsize=11, ha='center')

# Bottom: energy plot
ax2.set_xlim(0, n_steps * dt)
e_max = max(E_total.max(), max(K_traj.max(), U_traj.max())) * 1.2
ax2.set_ylim(0, e_max)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Energy (J)')
ax2.set_title('Energy Conservation: K + U = constant')

t_arr = np.arange(n_steps) * dt

line_K, = ax2.plot([], [], 'g-', linewidth=2, label='Kinetic Energy K')
line_U, = ax2.plot([], [], 'r-', linewidth=2, label='Potential Energy U')
line_E, = ax2.plot([], [], 'k--', linewidth=2, label='Total Energy E')
ax2.legend(loc='upper right', fontsize=10)

def init_anim():
    test_dot.set_data([], [])
    test_label.set_text('')
    line_K.set_data([], [])
    line_U.set_data([], [])
    line_E.set_data([], [])
    return test_dot, test_label, line_K, line_U, line_E

def animate_energy(frame):
    idx = frame * 4  # skip frames for speed
    if idx >= n_steps:
        idx = n_steps - 1
    
    # Update test charge position
    test_dot.set_data([x_traj[idx]], [0])
    test_label.set_position((x_traj[idx], 0.4))
    test_label.set_text(f'v = {v_traj[idx]:.2f} m/s')
    
    # Update energy plots
    line_K.set_data(t_arr[:idx], K_traj[:idx])
    line_U.set_data(t_arr[:idx], U_traj[:idx])
    line_E.set_data(t_arr[:idx], E_total[:idx])
    
    return test_dot, test_label, line_K, line_U, line_E

anim_energy = FuncAnimation(fig_anim, animate_energy, init_func=init_anim,
                            frames=n_steps // 4, interval=40, blit=False)

plt.tight_layout()
plt.close(fig_anim)
HTML(anim_energy.to_jshtml())

---
## 7. Interactive: Energy Conservation Problem Solver

Use this interactive tool to explore energy conservation for a charge moving in the field of a fixed charge. Adjust the initial speed and position to see how kinetic and potential energy trade off.

In [ ]:
# ============================================================
# Interactive: Energy Conservation Explorer
# ============================================================

def energy_explorer(Q_fixed_uC=5.0, q_test_uC=1.0, r_start=3.0, v_start=-1.5):
    """
    Visualize energy conservation for a test charge in a Coulomb field.
    """
    Q_f = Q_fixed_uC * 1e-6
    q_t = q_test_uC * 1e-6
    m = 1e-6  # artificial mass
    
    # Initial energies
    K_init = 0.5 * m * v_start**2
    U_init = k_e * q_t * Q_f / abs(r_start)
    E_tot = K_init + U_init
    
    # Plot U(r) and total energy line
    r_arr = np.linspace(0.2, 5.0, 500)
    U_arr = k_e * q_t * Q_f / r_arr
    K_arr = E_tot - U_arr  # K = E_tot - U
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(r_arr, U_arr, 'r-', linewidth=2.5, label='Potential Energy U(r)')
    ax.axhline(y=E_tot, color='black', linewidth=2, linestyle='--', label=f'Total Energy E = {E_tot:.6f} J')
    
    # Shade the kinetic energy region
    K_positive = np.where(K_arr > 0, K_arr, 0)
    ax.fill_between(r_arr, U_arr, E_tot, where=(K_arr > 0),
                    alpha=0.3, color='green', label='Kinetic Energy K = E - U')
    
    # Mark the starting position
    ax.plot(abs(r_start), U_init, 'go', markersize=12, zorder=10)
    ax.plot(abs(r_start), E_tot, 'ko', markersize=8, zorder=10)
    ax.annotate(f'Start\nr = {abs(r_start):.1f} m\nK = {K_init:.6f} J\nU = {U_init:.4f} J',
               xy=(abs(r_start), U_init),
               xytext=(abs(r_start) + 0.5, U_init + (E_tot - U_init) * 0.5),
               fontsize=10, arrowprops=dict(arrowstyle='->', color='green'))
    
    # Find turning point (where K = 0)
    if q_t * Q_f > 0:  # repulsive
        r_turn = k_e * q_t * Q_f / E_tot if E_tot > 0 else None
        if r_turn and 0.2 < r_turn < 5:
            ax.axvline(x=r_turn, color='purple', linestyle=':', linewidth=1.5)
            ax.text(r_turn + 0.1, E_tot * 0.5,
                    f'Closest approach\nr = {r_turn:.2f} m',
                    fontsize=10, color='purple')
    
    ax.set_xlabel('Distance r (m)', fontsize=12)
    ax.set_ylabel('Energy (J)', fontsize=12)
    ax.set_title('Energy Diagram: Conservation of Energy', fontsize=13)
    ax.legend(fontsize=10, loc='best')
    ax.set_xlim(0.2, 5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Initial: K = {K_init:.6f} J,  U = {U_init:.6f} J,  E_total = {E_tot:.6f} J")
    if q_t * Q_f > 0 and E_tot > 0:
        r_closest = k_e * q_t * Q_f / E_tot
        print(f"Closest approach distance: r_min = kqQ/E = {r_closest:.4f} m")

style = {'description_width': 'initial'}
layout = Layout(width='500px')

interact(energy_explorer,
         Q_fixed_uC=FloatSlider(min=1, max=10, step=0.5, value=5.0,
                                description='Q fixed (uC):', style=style, layout=layout),
         q_test_uC=FloatSlider(min=0.5, max=5, step=0.5, value=1.0,
                               description='q test (uC):', style=style, layout=layout),
         r_start=FloatSlider(min=0.5, max=4.5, step=0.25, value=3.0,
                             description='Start distance (m):', style=style, layout=layout),
         v_start=FloatSlider(min=-3, max=0, step=0.25, value=-1.5,
                             description='Init. velocity (m/s):', style=style, layout=layout));

---
## 8. Potential from Multiple Charges: Landscape Explorer

Let us combine everything: view the potential landscape from multiple charges with equipotentials and field lines together.

In [ ]:
# ============================================================
# Interactive: Multi-Charge Potential with 3D + Contour Side-by-Side
# ============================================================

def full_potential_explorer(q1=3.0, x1=-1.5, y1=0.0,
                            q2=-3.0, x2=1.5, y2=0.0,
                            q3=0.0, x3=0.0, y3=2.0):
    """
    Full potential landscape: 3D surface + 2D contour map side by side.
    """
    charges = []
    if q1 != 0:
        charges.append((q1 * 1e-6, x1, y1))
    if q2 != 0:
        charges.append((q2 * 1e-6, x2, y2))
    if q3 != 0:
        charges.append((q3 * 1e-6, x3, y3))
    
    if len(charges) == 0:
        print("Set at least one charge to a nonzero value.")
        return
    
    gx = np.linspace(-4, 4, 150)
    gy = np.linspace(-4, 4, 150)
    X, Y = np.meshgrid(gx, gy)
    
    V = V_total(charges, X, Y)
    V_clip = np.clip(V, -3e4, 3e4)
    
    fig = plt.figure(figsize=(16, 7))
    
    # --- Left: 3D surface ---
    ax1 = fig.add_subplot(121, projection='3d')
    surf = ax1.plot_surface(X, Y, V_clip, cmap='coolwarm',
                            alpha=0.8, rstride=3, cstride=3,
                            linewidth=0.1, edgecolor='gray')
    ax1.set_xlabel('x (m)')
    ax1.set_ylabel('y (m)')
    ax1.set_zlabel('V (V)')
    ax1.set_title('3D Potential Landscape')
    ax1.view_init(elev=25, azim=45)
    
    # --- Right: 2D contour + field lines ---
    ax2 = fig.add_subplot(122)
    
    V_max = min(np.percentile(np.abs(V_clip), 90), 3e4)
    levels = np.linspace(-V_max, V_max, 25)
    
    ax2.contourf(X, Y, V_clip, levels=50, cmap='coolwarm', alpha=0.4)
    cs = ax2.contour(X, Y, V_clip, levels=levels, cmap='coolwarm',
                     linewidths=1, alpha=0.7)
    
    Ex, Ey = E_field_total(charges, X, Y)
    E_mag = np.sqrt(Ex**2 + Ey**2)
    lw = 1.5 * np.log1p(E_mag) / np.log1p(E_mag).max()
    ax2.streamplot(X, Y, Ex, Ey, color='gray', density=1.5,
                   linewidth=lw, arrowsize=1)
    
    for q, xq, yq in charges:
        color = 'red' if q > 0 else 'blue'
        ax2.plot(xq, yq, 'o', color=color, markersize=16,
                markeredgecolor='black', markeredgewidth=2, zorder=10)
        sign = '+' if q > 0 else '-'
        ax2.text(xq, yq, sign, ha='center', va='center',
                fontsize=13, fontweight='bold', color='white', zorder=11)
    
    ax2.set_xlim(-4, 4)
    ax2.set_ylim(-4, 4)
    ax2.set_aspect('equal')
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('y (m)')
    ax2.set_title('Equipotentials + Field Lines')
    ax2.grid(True, alpha=0.15)
    
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
layout = Layout(width='420px')

interact(full_potential_explorer,
         q1=FloatSlider(min=-5, max=5, step=0.5, value=3.0,
                        description='q1 (uC):', style=style, layout=layout),
         x1=FloatSlider(min=-3, max=3, step=0.25, value=-1.5,
                        description='x1:', style=style, layout=layout),
         y1=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y1:', style=style, layout=layout),
         q2=FloatSlider(min=-5, max=5, step=0.5, value=-3.0,
                        description='q2 (uC):', style=style, layout=layout),
         x2=FloatSlider(min=-3, max=3, step=0.25, value=1.5,
                        description='x2:', style=style, layout=layout),
         y2=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='y2:', style=style, layout=layout),
         q3=FloatSlider(min=-5, max=5, step=0.5, value=0.0,
                        description='q3 (uC) [0=off]:', style=style, layout=layout),
         x3=FloatSlider(min=-3, max=3, step=0.25, value=0.0,
                        description='x3:', style=style, layout=layout),
         y3=FloatSlider(min=-3, max=3, step=0.25, value=2.0,
                        description='y3:', style=style, layout=layout));

In [ ]:
# ============================================================
# Worked Example 3: Energy Conservation -- Speed of a charge
# ============================================================

print("=" * 60)
print("WORKED EXAMPLE 3: Finding Speed Using Energy Conservation")
print("=" * 60)
print()
print("Problem: A proton (m = 1.67e-27 kg, q = +e) is released")
print("from rest at r_A = 0.10 m from a fixed charge Q = +5 nC.")
print("What is the proton's speed when it reaches r_B = 0.50 m?")
print()

m_p = 1.67e-27  # proton mass, kg
q_p = e_charge   # proton charge, C
Q_f = 5e-9       # fixed charge, C
rA = 0.10        # m
rB = 0.50        # m
vA = 0           # released from rest

VA = k_e * Q_f / rA
VB = k_e * Q_f / rB

# Energy conservation: (1/2)m v_A^2 + q V_A = (1/2)m v_B^2 + q V_B
# 0 + q V_A = (1/2) m v_B^2 + q V_B
# v_B = sqrt(2 q (V_A - V_B) / m)

delta_V = VA - VB
vB = np.sqrt(2 * q_p * delta_V / m_p)

print(f"Solution using energy conservation:")
print(f"  K_A + U_A = K_B + U_B")
print(f"  0 + q V_A = (1/2) m v_B^2 + q V_B")
print(f"  v_B = sqrt(2 q (V_A - V_B) / m)")
print()
print(f"  V_A = kQ/r_A = {VA:.2f} V")
print(f"  V_B = kQ/r_B = {VB:.2f} V")
print(f"  V_A - V_B = {delta_V:.2f} V")
print()
print(f"  v_B = sqrt(2 x {q_p:.3e} x {delta_V:.2f} / {m_p:.2e})")
print(f"       = {vB:.2e} m/s")
print(f"       = {vB/1e3:.2f} km/s")
print()
print(f"  This is about {vB/3e8*100:.4f}% of the speed of light.")
print(f"  (Non-relativistic, so our calculation is valid.)")

---
## 9. Gradient Relationship: E = -grad(V)

Let us visualize the gradient relationship directly. The electric field at any point equals the negative gradient of the potential. Steeper potential slopes mean stronger electric fields.

In [ ]:
# ============================================================
# Visualization: E = -grad(V) demonstrated along a line
# ============================================================

def gradient_demo(Q=5.0, y_slice=0.5):
    """
    Show V(x) and E_x(x) along a horizontal line y = y_slice.
    Demonstrates E_x = -dV/dx.
    Q in microcoulombs.
    """
    Q_C = Q * 1e-6
    charges = [(Q_C, 0.0, 0.0)]
    
    x_arr = np.linspace(-3, 3, 500)
    y_val = y_slice
    
    # Calculate V along the slice (vectorized)
    y_arr_full = np.full_like(x_arr, y_val)
    V_arr = V_total(charges, x_arr, y_arr_full)
    
    # Calculate E_x along the slice (vectorized)
    Ex_arr, _ = E_field_total(charges, x_arr, y_arr_full)
    
    # Numerical gradient for comparison
    dVdx = -np.gradient(V_arr, x_arr)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Top: V(x)
    V_clip = np.clip(V_arr, -1e5, 1e5)
    ax1.plot(x_arr, V_clip / 1e3, 'b-', linewidth=2.5, label='V(x)')
    ax1.axhline(y=0, color='gray', linewidth=0.5)
    ax1.set_ylabel('V (kV)', fontsize=12)
    ax1.set_title(f'Potential and Field along y = {y_val:.1f} m (Q = {Q:.1f} uC at origin)',
                 fontsize=13)
    ax1.legend(fontsize=12)
    y_lim = min(200, abs(V_clip).max() / 1e3 * 1.1)
    ax1.set_ylim(-y_lim if Q < 0 else -20, y_lim if Q > 0 else 20)
    
    # Bottom: E_x(x) vs -dV/dx
    Ex_clip = np.clip(Ex_arr, -1e6, 1e6)
    dVdx_clip = np.clip(dVdx, -1e6, 1e6)
    ax2.plot(x_arr, Ex_clip / 1e3, 'r-', linewidth=2.5, label=r'$E_x$ (from Coulomb\'s law)')
    ax2.plot(x_arr, dVdx_clip / 1e3, 'k--', linewidth=1.5, alpha=0.7,
             label=r'$-dV/dx$ (numerical gradient)')
    ax2.axhline(y=0, color='gray', linewidth=0.5)
    ax2.set_xlabel('x (m)', fontsize=12)
    ax2.set_ylabel('E_x (kN/C)', fontsize=12)
    ax2.set_title(r'Electric Field: $E_x = -dV/dx$ (should overlap!)', fontsize=13)
    ax2.legend(fontsize=11)
    ex_lim = min(500, abs(Ex_clip).max() / 1e3 * 0.5)
    ax2.set_ylim(-ex_lim, ex_lim)
    
    plt.tight_layout()
    plt.show()
    
    print("The red and black curves overlap, confirming E_x = -dV/dx.")
    print("Where V has a steep slope, E is large. Where V is flat, E is small.")

interact(gradient_demo,
         Q=FloatSlider(min=-5, max=5, step=0.5, value=5.0,
                       description='Q (uC):', style={'description_width': 'initial'},
                       layout=Layout(width='500px')),
         y_slice=FloatSlider(min=0.2, max=2.5, step=0.1, value=0.5,
                             description='y-slice (m):', style={'description_width': 'initial'},
                             layout=Layout(width='500px')));

---
## 10. Summary

### Key Equations from This Week

| Concept | Equation |
|---------|----------|
| Potential from point charge | $V = k_e Q / r$ |
| Superposition of potentials | $V_{\text{total}} = \sum_i k_e q_i / r_i$ |
| Potential energy | $U = q_0 V = k_e q_1 q_2 / r$ |
| Work by electric force | $W = q_0 (V_A - V_B) = -q_0 \Delta V$ |
| E-V relationship | $\vec{E} = -\nabla V$ |
| Energy conservation | $K_A + U_A = K_B + U_B$ |

### Key Concepts
- Electric potential $V$ is a scalar field -- much simpler than the vector field $\vec{E}$
- Potential from positive charges is positive; from negative charges, negative
- $V$ drops off as $1/r$ (slower than $E$ which drops as $1/r^2$)
- Equipotential surfaces are perpendicular to field lines
- The electric force is conservative: work depends only on endpoints, not the path
- Energy conservation ($K + U = \text{const}$) is a powerful problem-solving tool

---
## Problem Set

> **For each problem: Identify the configuration → Choose the law → Write the equation → Predict → Verify**

Work through the following problems on electric potential, potential energy, and energy conservation. Problems are graded by difficulty:
- **L1 (Basic):** Single-concept, direct application of formulas
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

Use $k_e = 8.99 \times 10^9$ N·m²/C², $e = 1.60 \times 10^{-19}$ C.

---

### L1 — Basic Problems

**P1.** A point charge $Q = +3.0\;\mu$C is at the origin. Calculate the electric potential at a distance $r = 0.50$ m from the charge.

<details><summary>Answer</summary>5.39 × 10⁴ V</details>

In [ ]:
# ✏️ [P1] Your solution here


**P2.** Two point charges, $q_1 = +5.0\;\mu$C at $(-0.30, 0)$ m and $q_2 = -3.0\;\mu$C at $(+0.30, 0)$ m, are fixed in place. Find the electric potential at the origin.

<details><summary>Answer</summary>5.99 × 10⁴ V</details>

In [ ]:
# ✏️ [P2] Your solution here


**P3.** A test charge $q_0 = +2.0\;\mu$C is moved from a point where $V_A = 450$ V to a point where $V_B = 300$ V. Calculate the work done by the electric field on the charge.

<details><summary>Answer</summary>3.00 × 10⁻⁴ J</details>

In [ ]:
# ✏️ [P3] Your solution here


**P4.** An electron is accelerated from rest through a potential difference of $\Delta V = 500$ V. (a) What kinetic energy does it gain, in eV and in joules? (b) What is its final speed?

<details><summary>Answer</summary>(a) 500 eV = 8.00 × 10⁻¹⁷ J, (b) 1.33 × 10⁷ m/s</details>

In [ ]:
# ✏️ [P4] Your solution here


### L2 -- Intermediate Problems

**P5.** Three point charges are placed at the vertices of an equilateral triangle with side length $a = 0.25$ m: $q_1 = +4.0\;\mu$C, $q_2 = -2.0\;\mu$C, and $q_3 = +6.0\;\mu$C. Calculate the total electric potential energy of this charge configuration.

<details><summary>Answer</summary>−0.360 J</details>

In [ ]:
# ✏️ [P5] Your solution here


**P6.** A proton is released from rest at $r_i = 0.50$ m from a fixed point charge $Q = +4.0$ nC. Using energy conservation, find the proton's speed when it has moved to $r_f = 2.0$ m.

<details><summary>Answer</summary>3.22 × 10⁴ m/s</details>

In [ ]:
# ✏️ [P6] Your solution here


**P7.** A uniform electric field of magnitude $E = 4000$ V/m points in the $+x$ direction. A charge $q = +3.0\;\mu$C is moved from point $A = (0.10, 0)$ m to point $B = (0.40, 0.30)$ m. Find the potential difference $V_A - V_B$ and the work done by the field.

<details><summary>Answer</summary>V_A − V_B = 1200 V, W = 3.60 × 10⁻³ J</details>

In [ ]:
# ✏️ [P7] Your solution here


**P8.** Two isolated conducting spheres have radii $R_1 = 5.0$ cm and $R_2 = 10.0$ cm and carry charges $Q_1 = +6.0\;\mu$C and $Q_2 = 0$. They are then connected by a thin conducting wire until equilibrium is reached. Find the final charge on each sphere and their common potential. (Hint: charge distributes so that both spheres reach the same potential.)

<details><summary>Answer</summary>Q₁' = 2.0 μC, Q₂' = 4.0 μC, V = 3.60 × 10⁵ V</details>

In [ ]:
# ✏️ [P8] Your solution here


### L3 -- Challenge Problems

**P9.** An alpha particle ($q = +2e$, $m = 6.64 \times 10^{-27}$ kg) is fired with kinetic energy $K_0 = 5.0$ MeV directly toward a gold nucleus ($Q = +79e$), which is held fixed. Find the distance of closest approach. (This is the classic Rutherford scattering problem.)

<details><summary>Answer</summary>4.55 × 10⁻¹⁴ m</details>

In [ ]:
# ✏️ [P9] Your solution here


**P10.** The electric potential in a region of space is given by $V(x) = 100 - 30x + 5x^2$ (in volts, with $x$ in meters). (a) Find the electric field $E_x(x)$ as a function of $x$. (b) At what value of $x$ is the electric field zero? (c) What is the electric field at $x = 0$ and $x = 6.0$ m?

<details><summary>Answer</summary>(a) E_x = 30 − 10x V/m, (b) x = 3.0 m, (c) E(0) = 30 V/m, E(6) = −30 V/m</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## Bridge to Next Week

We have now covered the foundations of electrostatics:
- Week 1: Charges, forces ($\vec{F}$), and fields ($\vec{E}$)
- Week 2: Potential ($V$), energy ($U$), and the E-V relationship

Next week, we will study **Gauss's Law**, one of the four Maxwell equations. Gauss's Law provides:

- A beautiful relationship between electric flux and enclosed charge
- A powerful method to calculate $\vec{E}$ for highly symmetric charge distributions
- Deep insight into why the electric field behaves the way it does

**Reading:** Think about what "flux" means intuitively -- how much of a field "flows through" a surface. We will make this precise with integrals next week.

---
*End of Week 02 -- Physics II (PHY102)*